# DHIS2 Water Facility Setup

This notebook sets up the Water Facility Tracked Entity in DHIS2, designed to sync with Sunbird RC.

## Prerequisites
- DHIS2 instance running at http://localhost:9090
- Admin credentials (default: admin/district)

## Setup Flow
1. Create Option Sets
2. Create Tracked Entity Attributes
3. Create Tracked Entity Type
4. Create Tracker Program
5. Import Organisation Units

## 0. Setup and Authentication

In [1]:
import requests
import json
import pandas as pd
from IPython.display import display, JSON, Markdown

# Configuration
BASE_URL = "http://localhost:9090/api"
AUTH = ("admin", "district")

HEADERS = {
    "Content-Type": "application/json",
    "Accept": "application/json"
}

# Helper function for API calls
def dhis2_post(endpoint, data):
    response = requests.post(
        f"{BASE_URL}/{endpoint}",
        auth=AUTH,
        headers=HEADERS,
        json=data
    )
    return response

def dhis2_get(endpoint):
    response = requests.get(
        f"{BASE_URL}/{endpoint}",
        auth=AUTH,
        headers=HEADERS
    )
    return response

print(f"DHIS2 API: {BASE_URL}")

DHIS2 API: http://localhost:9090/api


## 1. Check DHIS2 Health

In [2]:
response = dhis2_get("system/info")

if response.status_code == 200:
    info = response.json()
    print(f"DHIS2 Version: {info.get('version')}")
    print(f"Database: {info.get('databaseInfo', {}).get('name')}")
    print(f"Server Date: {info.get('serverDate')}")
else:
    print(f"Error: {response.status_code}")
    print(response.text)

DHIS2 Version: 2.40.4.1
Database: dhis2
Server Date: 2026-04-27T06:00:23.271


## 2. Create Option Sets

Create option sets for dropdown fields.

In [3]:
# Define all option sets with their options
OPTION_SETS = [
    {
        "name": "Water Point Type",
        "code": "WATER_POINT_TYPE",
        "valueType": "TEXT",
        "options": [
            {"name": "Protected dug well", "code": "PDW"},
            {"name": "Unprotected dug well", "code": "UDW"},
            {"name": "Tube well or borehole", "code": "TWB"},
            {"name": "Protected spring", "code": "PS"},
            {"name": "Unprotected spring", "code": "US"},
            {"name": "Piped water into dwelling/plot/yard", "code": "PWD"},
            {"name": "Public tap/standpipe", "code": "PTS"},
            {"name": "Unequipped borehole", "code": "UEB"},
            {"name": "Rainwater (harvesting)", "code": "RWH"},
            {"name": "Sand/Sub-surface dam", "code": "SSD"},
            {"name": "Other", "code": "OTH"}
        ]
    },
    {
        "name": "Extraction Type",
        "code": "EXTRACTION_TYPE",
        "valueType": "TEXT",
        "options": [
            {"name": "Manual", "code": "MANUAL"},
            {"name": "Electrical", "code": "ELECTRICAL"},
            {"name": "Solar", "code": "SOLAR"},
            {"name": "Other", "code": "OTHER"}
        ]
    },
    {
        "name": "Pump Type",
        "code": "PUMP_TYPE",
        "valueType": "TEXT",
        "options": [
            {"name": "Afridev", "code": "AFRIDEV"},
            {"name": "Consallen", "code": "CONSALLEN"},
            {"name": "India Mark", "code": "INDIA_MARK"},
            {"name": "Kardia", "code": "KARDIA"},
            {"name": "Rope pump", "code": "ROPE_PUMP"},
            {"name": "Vergnet", "code": "VERGNET"},
            {"name": "Other", "code": "OTHER"}
        ]
    },
    {
        "name": "Installer Type",
        "code": "INSTALLER_TYPE",
        "valueType": "TEXT",
        "options": [
            {"name": "Government", "code": "GOVERNMENT"},
            {"name": "NGO", "code": "NGO"},
            {"name": "Private", "code": "PRIVATE"},
            {"name": "Other", "code": "OTHER"}
        ]
    },
    {
        "name": "Owner Type",
        "code": "OWNER_TYPE",
        "valueType": "TEXT",
        "options": [
            {"name": "Community", "code": "COMMUNITY"},
            {"name": "Private Individual", "code": "PRIVATE_INDIVIDUAL"},
            {"name": "School", "code": "SCHOOL"},
            {"name": "NGO", "code": "NGO"},
            {"name": "Health Facility", "code": "HEALTH_FACILITY"},
            {"name": "Other institution", "code": "OTHER_INSTITUTION"},
            {"name": "CBO", "code": "CBO"},
            {"name": "Private", "code": "PRIVATE"},
            {"name": "Unknown", "code": "UNKNOWN"},
            {"name": "Other", "code": "OTHER"}
        ]
    },
    {
        "name": "Sync Status",
        "code": "SYNC_STATUS",
        "valueType": "TEXT",
        "options": [
            {"name": "Pending", "code": "PENDING"},
            {"name": "Synced", "code": "SYNCED"},
            {"name": "Failed", "code": "FAILED"}
        ]
    }
]

print(f"Defined {len(OPTION_SETS)} option sets")
for os in OPTION_SETS:
    print(f"  - {os['name']}: {len(os['options'])} options")

Defined 6 option sets
  - Water Point Type: 11 options
  - Extraction Type: 4 options
  - Pump Type: 7 options
  - Installer Type: 4 options
  - Owner Type: 10 options
  - Sync Status: 3 options


In [4]:
# Create option sets using metadata endpoint
created_option_sets = {}

for option_set in OPTION_SETS:
    # Prepare options with sort order
    options = []
    for i, opt in enumerate(option_set["options"]):
        options.append({
            "name": opt["name"],
            "code": opt["code"],
            "sortOrder": i + 1
        })
    
    # Create metadata payload
    metadata = {
        "optionSets": [
            {
                "name": option_set["name"],
                "code": option_set["code"],
                "valueType": option_set["valueType"],
                "options": options
            }
        ]
    }
    
    response = dhis2_post("metadata", metadata)
    
    if response.status_code in [200, 201]:
        result = response.json()
        status = result.get("status")
        
        if status == "OK":
            # Get the created option set ID
            get_response = dhis2_get(f"optionSets?filter=code:eq:{option_set['code']}&fields=id,name,code")
            if get_response.status_code == 200:
                os_data = get_response.json()
                if os_data.get("optionSets"):
                    os_id = os_data["optionSets"][0]["id"]
                    created_option_sets[option_set["code"]] = os_id
                    print(f"✓ Created: {option_set['name']} (ID: {os_id})")
        else:
            print(f"✗ Failed: {option_set['name']} - {result}")
    else:
        print(f"✗ Error: {option_set['name']} - {response.status_code}")
        print(response.text)

print(f"\nCreated {len(created_option_sets)} option sets")

✗ Error: Water Point Type - 409
{"httpStatus":"Conflict","httpStatusCode":409,"status":"WARNING","message":"One or more errors occurred, please see full details in import report.","response":{"responseType":"ImportReport","status":"ERROR","stats":{"created":0,"updated":0,"deleted":0,"ignored":2,"total":2},"typeReports":[{"klass":"org.hisp.dhis.option.OptionSet","stats":{"created":0,"updated":0,"deleted":0,"ignored":2,"total":2},"objectReports":[{"klass":"org.hisp.dhis.option.OptionSet","index":0,"uid":"i5zxbgtJkUJ","errorReports":[{"message":"Invalid reference Protected dug well [hNI5g8MDaxQ] (Option) on object Water Point Type [i5zxbgtJkUJ] (OptionSet) for association `option`","mainKlass":"org.hisp.dhis.option.OptionSet","mainId":"i5zxbgtJkUJ","errorKlass":"org.hisp.dhis.option.Option","errorProperty":"option","errorProperties":["Protected dug well [hNI5g8MDaxQ] (Option)","Water Point Type [i5zxbgtJkUJ] (OptionSet)","option"],"errorCode":"E5002"},{"message":"Invalid reference Unprote

## 3. Create Tracked Entity Attributes

In [ ]:
# Define tracked entity attributes
# Note: optionSet will be added where applicable using the IDs from created_option_sets

ATTRIBUTES = [
    # Sync fields (filled by adapter)
    {
        "name": "Sunbird OSID",
        "shortName": "osid",
        "code": "SUNBIRD_OSID",
        "valueType": "TEXT",
        "aggregationType": "NONE",
        "unique": True,
        "searchable": True
    },
    {
        "name": "Water Facility ID",
        "shortName": "wfId",
        "code": "WF_ID",
        "valueType": "TEXT",
        "aggregationType": "NONE",
        "unique": True,
        "searchable": True
    },
    {
        "name": "Sync Status",
        "shortName": "syncStatus",
        "code": "SYNC_STATUS_ATTR",
        "valueType": "TEXT",
        "aggregationType": "NONE",
        "optionSetCode": "SYNC_STATUS"
    },
    # Location fields
    {
        "name": "Geo Code",
        "shortName": "geoCode",
        "code": "GEO_CODE",
        "valueType": "TEXT",
        "aggregationType": "NONE",
        "unique": True,
        "searchable": True
    },
    {
        "name": "County",
        "shortName": "county",
        "code": "COUNTY",
        "valueType": "TEXT",
        "aggregationType": "NONE",
        "searchable": True
    },
    {
        "name": "District",
        "shortName": "district",
        "code": "DISTRICT",
        "valueType": "TEXT",
        "aggregationType": "NONE",
        "searchable": True
    },
    {
        "name": "Community",
        "shortName": "community",
        "code": "COMMUNITY",
        "valueType": "TEXT",
        "aggregationType": "NONE"
    },
    # Facility details
    {
        "name": "Water Point Type",
        "shortName": "waterPointType",
        "code": "WATER_POINT_TYPE_ATTR",
        "valueType": "TEXT",
        "aggregationType": "NONE",
        "optionSetCode": "WATER_POINT_TYPE"
    },
    {
        "name": "Extraction Type",
        "shortName": "extractionType",
        "code": "EXTRACTION_TYPE_ATTR",
        "valueType": "TEXT",
        "aggregationType": "NONE",
        "optionSetCode": "EXTRACTION_TYPE"
    },
    {
        "name": "Pump Type",
        "shortName": "pumpType",
        "code": "PUMP_TYPE_ATTR",
        "valueType": "TEXT",
        "aggregationType": "NONE",
        "optionSetCode": "PUMP_TYPE"
    },
    {
        "name": "Number of Taps",
        "shortName": "numTaps",
        "code": "NUM_TAPS",
        "valueType": "INTEGER",
        "aggregationType": "SUM"
    },
    {
        "name": "Has Depth Info",
        "shortName": "hasDepthInfo",
        "code": "HAS_DEPTH_INFO",
        "valueType": "BOOLEAN",
        "aggregationType": "NONE"
    },
    {
        "name": "Depth (metres)",
        "shortName": "depthMetres",
        "code": "DEPTH_METRES",
        "valueType": "NUMBER",
        "aggregationType": "AVERAGE"
    },
    # Ownership & Installation
    {
        "name": "Installer",
        "shortName": "installer",
        "code": "INSTALLER",
        "valueType": "TEXT",
        "aggregationType": "NONE",
        "optionSetCode": "INSTALLER_TYPE"
    },
    {
        "name": "Owner",
        "shortName": "owner",
        "code": "OWNER",
        "valueType": "TEXT",
        "aggregationType": "NONE",
        "optionSetCode": "OWNER_TYPE"
    },
    {
        "name": "Funder",
        "shortName": "funder",
        "code": "FUNDER",
        "valueType": "TEXT",
        "aggregationType": "NONE"
    },
    {
        "name": "Photo URL",
        "shortName": "photoUrl",
        "code": "PHOTO_URL",
        "valueType": "URL",
        "aggregationType": "NONE"
    }
]

print(f"Defined {len(ATTRIBUTES)} tracked entity attributes")

In [ ]:
# Create tracked entity attributes
created_attributes = {}

for attr in ATTRIBUTES:
    # Build attribute payload
    payload = {
        "name": attr["name"],
        "shortName": attr["shortName"],
        "code": attr["code"],
        "valueType": attr["valueType"],
        "aggregationType": attr["aggregationType"]
    }
    
    # Add optional fields
    if attr.get("unique"):
        payload["unique"] = True
    
    # Link to option set if specified
    if attr.get("optionSetCode") and attr["optionSetCode"] in created_option_sets:
        payload["optionSet"] = {"id": created_option_sets[attr["optionSetCode"]]}
    
    # Create via metadata endpoint
    metadata = {"trackedEntityAttributes": [payload]}
    response = dhis2_post("metadata", metadata)
    
    if response.status_code in [200, 201]:
        result = response.json()
        if result.get("status") == "OK":
            # Get the created attribute ID
            get_response = dhis2_get(f"trackedEntityAttributes?filter=code:eq:{attr['code']}&fields=id,name,code")
            if get_response.status_code == 200:
                attr_data = get_response.json()
                if attr_data.get("trackedEntityAttributes"):
                    attr_id = attr_data["trackedEntityAttributes"][0]["id"]
                    created_attributes[attr["code"]] = {
                        "id": attr_id,
                        "name": attr["name"],
                        "searchable": attr.get("searchable", False)
                    }
                    print(f"✓ Created: {attr['name']} (ID: {attr_id})")
        else:
            print(f"✗ Failed: {attr['name']} - {result}")
    else:
        print(f"✗ Error: {attr['name']} - {response.status_code}")
        print(response.text)

print(f"\nCreated {len(created_attributes)} attributes")

## 4. Create Tracked Entity Type

In [ ]:
# Build tracked entity type attributes list
te_type_attributes = []
for i, (code, attr_info) in enumerate(created_attributes.items()):
    te_type_attributes.append({
        "trackedEntityAttribute": {"id": attr_info["id"]},
        "displayInList": code in ["SUNBIRD_OSID", "WF_ID", "SYNC_STATUS_ATTR", "GEO_CODE", "COUNTY", "DISTRICT", "COMMUNITY", "WATER_POINT_TYPE_ATTR"],
        "searchable": attr_info.get("searchable", False),
        "sortOrder": i + 1
    })

# Create tracked entity type
tracked_entity_type = {
    "name": "Water Facility",
    "shortName": "WaterFacility",
    "code": "WATER_FACILITY",
    "description": "Water facilities with Sunbird RC integration",
    "featureType": "POINT",
    "trackedEntityTypeAttributes": te_type_attributes
}

metadata = {"trackedEntityTypes": [tracked_entity_type]}
response = dhis2_post("metadata", metadata)

if response.status_code in [200, 201]:
    result = response.json()
    if result.get("status") == "OK":
        # Get the created type ID
        get_response = dhis2_get("trackedEntityTypes?filter=code:eq:WATER_FACILITY&fields=id,name,code")
        if get_response.status_code == 200:
            type_data = get_response.json()
            if type_data.get("trackedEntityTypes"):
                te_type_id = type_data["trackedEntityTypes"][0]["id"]
                print(f"✓ Created Tracked Entity Type: Water Facility (ID: {te_type_id})")
    else:
        print(f"✗ Failed: {result}")
else:
    print(f"✗ Error: {response.status_code}")
    print(response.text)

## 5. Import Organisation Units

First, let's check existing org units and get the root org unit.

In [ ]:
# Check existing organisation units
response = dhis2_get("organisationUnits?filter=level:eq:1&fields=id,name,code,level")

if response.status_code == 200:
    org_units = response.json()
    if org_units.get("organisationUnits"):
        root_ou = org_units["organisationUnits"][0]
        ROOT_OU_ID = root_ou["id"]
        print(f"Root Organisation Unit: {root_ou['name']} (ID: {ROOT_OU_ID})")
    else:
        print("No root organisation unit found. Will create Liberia as root.")
        ROOT_OU_ID = None
else:
    print(f"Error: {response.status_code}")
    ROOT_OU_ID = None

In [ ]:
# Load organisation units from CSV
import os

csv_path = "org_units_sample.csv"

if os.path.exists(csv_path):
    org_df = pd.read_csv(csv_path)
    print(f"Loaded {len(org_df)} organisation units from {csv_path}")
    display(org_df)
else:
    print(f"CSV file not found: {csv_path}")
    print("Please create the CSV file first, then re-run this cell.")

In [ ]:
# Create organisation units from CSV
# This creates them level by level (Country -> Counties -> Districts -> Clans)

created_org_units = {}

if 'org_df' in dir() and not org_df.empty:
    # Sort by level to create parents first
    org_df_sorted = org_df.sort_values('level')
    
    for _, row in org_df_sorted.iterrows():
        # Determine parent
        if pd.isna(row.get('parent_code')) or row['parent_code'] == '':
            parent_id = None
        else:
            parent_id = created_org_units.get(row['parent_code'])
        
        # Build org unit payload
        org_unit = {
            "name": row['name'],
            "shortName": row['short_name'],
            "code": row['code'],
            "openingDate": "2020-01-01"
        }
        
        if parent_id:
            org_unit["parent"] = {"id": parent_id}
        
        # Create via metadata endpoint
        metadata = {"organisationUnits": [org_unit]}
        response = dhis2_post("metadata", metadata)
        
        if response.status_code in [200, 201]:
            result = response.json()
            if result.get("status") == "OK":
                # Get the created org unit ID
                get_response = dhis2_get(f"organisationUnits?filter=code:eq:{row['code']}&fields=id,name,code")
                if get_response.status_code == 200:
                    ou_data = get_response.json()
                    if ou_data.get("organisationUnits"):
                        ou_id = ou_data["organisationUnits"][0]["id"]
                        created_org_units[row['code']] = ou_id
                        level_indicator = "  " * (int(row['level']) - 1)
                        print(f"{level_indicator}✓ {row['name']} (ID: {ou_id})")
            else:
                print(f"✗ Failed: {row['name']} - {result}")
        else:
            print(f"✗ Error: {row['name']} - {response.status_code}")
    
    print(f"\nCreated {len(created_org_units)} organisation units")
else:
    print("No organisation units to create. Load CSV first.")

## 6. Create Tracker Program

In [ ]:
# Get all organisation units for program assignment
response = dhis2_get("organisationUnits?fields=id&paging=false")
all_org_units = []

if response.status_code == 200:
    ou_data = response.json()
    all_org_units = [{"id": ou["id"]} for ou in ou_data.get("organisationUnits", [])]
    print(f"Found {len(all_org_units)} organisation units for program assignment")

In [ ]:
# Build program tracked entity attributes
# Define which attributes are mandatory
MANDATORY_ATTRS = ["GEO_CODE", "COUNTY", "DISTRICT", "COMMUNITY", "WATER_POINT_TYPE_ATTR"]

program_attributes = []
for i, (code, attr_info) in enumerate(created_attributes.items()):
    program_attributes.append({
        "trackedEntityAttribute": {"id": attr_info["id"]},
        "displayInList": code in ["SUNBIRD_OSID", "WF_ID", "SYNC_STATUS_ATTR", "GEO_CODE", "COUNTY", "DISTRICT", "WATER_POINT_TYPE_ATTR"],
        "mandatory": code in MANDATORY_ATTRS,
        "searchable": attr_info.get("searchable", False),
        "sortOrder": i + 1
    })

# Create the tracker program
program = {
    "name": "Water Facility Registry",
    "shortName": "WF Registry",
    "code": "WF_REGISTRY",
    "programType": "WITH_REGISTRATION",
    "trackedEntityType": {"id": te_type_id},
    "displayFrontPageList": True,
    "featureType": "POINT",
    "onlyEnrollOnce": True,
    "programTrackedEntityAttributes": program_attributes,
    "organisationUnits": all_org_units if all_org_units else []
}

metadata = {"programs": [program]}
response = dhis2_post("metadata", metadata)

if response.status_code in [200, 201]:
    result = response.json()
    if result.get("status") == "OK":
        # Get the created program ID
        get_response = dhis2_get("programs?filter=code:eq:WF_REGISTRY&fields=id,name,code")
        if get_response.status_code == 200:
            prog_data = get_response.json()
            if prog_data.get("programs"):
                program_id = prog_data["programs"][0]["id"]
                print(f"✓ Created Program: Water Facility Registry (ID: {program_id})")
    else:
        print(f"✗ Failed: {result}")
        display(JSON(result))
else:
    print(f"✗ Error: {response.status_code}")
    print(response.text)

## 7. Verify Setup

In [ ]:
# Summary of created resources
print("=" * 60)
print("DHIS2 Water Facility Setup Summary")
print("=" * 60)

# Option Sets
response = dhis2_get("optionSets?filter=code:in:[WATER_POINT_TYPE,EXTRACTION_TYPE,PUMP_TYPE,INSTALLER_TYPE,OWNER_TYPE,SYNC_STATUS]&fields=id,name,code")
if response.status_code == 200:
    data = response.json()
    print(f"\nOption Sets: {len(data.get('optionSets', []))}")
    for os in data.get('optionSets', []):
        print(f"  - {os['name']} ({os['code']})")

# Tracked Entity Attributes
response = dhis2_get("trackedEntityAttributes?paging=false&fields=id,name,code")
if response.status_code == 200:
    data = response.json()
    attrs = [a for a in data.get('trackedEntityAttributes', []) if a['code'] in created_attributes]
    print(f"\nTracked Entity Attributes: {len(attrs)}")

# Tracked Entity Type
response = dhis2_get("trackedEntityTypes?filter=code:eq:WATER_FACILITY&fields=id,name,code")
if response.status_code == 200:
    data = response.json()
    print(f"\nTracked Entity Type: {len(data.get('trackedEntityTypes', []))}")
    for tet in data.get('trackedEntityTypes', []):
        print(f"  - {tet['name']} ({tet['code']})")

# Program
response = dhis2_get("programs?filter=code:eq:WF_REGISTRY&fields=id,name,code,programType")
if response.status_code == 200:
    data = response.json()
    print(f"\nPrograms: {len(data.get('programs', []))}")
    for prog in data.get('programs', []):
        print(f"  - {prog['name']} ({prog['code']}) - {prog['programType']}")

# Organisation Units
response = dhis2_get("organisationUnits?paging=false&fields=id,name,level")
if response.status_code == 200:
    data = response.json()
    print(f"\nOrganisation Units: {len(data.get('organisationUnits', []))}")
    levels = {}
    for ou in data.get('organisationUnits', []):
        level = ou.get('level', 0)
        levels[level] = levels.get(level, 0) + 1
    for level in sorted(levels.keys()):
        print(f"  - Level {level}: {levels[level]} units")

print("\n" + "=" * 60)
print("Setup complete! You can now:")
print("1. Go to Tracker Capture app")
print("2. Select an organisation unit")
print("3. Register new Water Facilities")
print("=" * 60)

## 8. Test: Create a Sample Water Facility

In [ ]:
# Get a leaf organisation unit (lowest level) for testing
response = dhis2_get("organisationUnits?filter=level:ge:3&fields=id,name,code&pageSize=1")

test_ou_id = None
if response.status_code == 200:
    data = response.json()
    if data.get('organisationUnits'):
        test_ou = data['organisationUnits'][0]
        test_ou_id = test_ou['id']
        print(f"Test Organisation Unit: {test_ou['name']} (ID: {test_ou_id})")
    else:
        # Fallback to any org unit
        response = dhis2_get("organisationUnits?fields=id,name&pageSize=1")
        if response.status_code == 200:
            data = response.json()
            if data.get('organisationUnits'):
                test_ou = data['organisationUnits'][0]
                test_ou_id = test_ou['id']
                print(f"Test Organisation Unit: {test_ou['name']} (ID: {test_ou_id})")

In [ ]:
# Create a test water facility
if test_ou_id and te_type_id:
    # Build attributes array
    test_attributes = [
        {"attribute": created_attributes["GEO_CODE"]["id"], "value": "TEST001"},
        {"attribute": created_attributes["COUNTY"]["id"], "value": "Montserrado"},
        {"attribute": created_attributes["DISTRICT"]["id"], "value": "Greater Monrovia"},
        {"attribute": created_attributes["COMMUNITY"]["id"], "value": "Congo Town"},
        {"attribute": created_attributes["WATER_POINT_TYPE_ATTR"]["id"], "value": "TWB"},
        {"attribute": created_attributes["SYNC_STATUS_ATTR"]["id"], "value": "PENDING"},
        {"attribute": created_attributes["EXTRACTION_TYPE_ATTR"]["id"], "value": "MANUAL"},
        {"attribute": created_attributes["INSTALLER"]["id"], "value": "NGO"},
        {"attribute": created_attributes["OWNER"]["id"], "value": "COMMUNITY"}
    ]
    
    # Create tracked entity instance
    tei_payload = {
        "trackedEntityType": te_type_id,
        "orgUnit": test_ou_id,
        "attributes": test_attributes,
        "geometry": {
            "type": "Point",
            "coordinates": [-10.7957, 6.3156]  # Monrovia coordinates
        }
    }
    
    response = dhis2_post("trackedEntityInstances", tei_payload)
    
    if response.status_code in [200, 201]:
        result = response.json()
        if result.get("status") == "OK" or result.get("response", {}).get("status") == "SUCCESS":
            tei_id = result.get("response", {}).get("importSummaries", [{}])[0].get("reference")
            print(f"✓ Created test Water Facility (TEI ID: {tei_id})")
            print(f"\nView in Tracker Capture:")
            print(f"  http://localhost:9090/dhis-web-tracker-capture/index.html")
        else:
            print(f"✗ Failed to create: {result}")
    else:
        print(f"✗ Error: {response.status_code}")
        print(response.text)
else:
    print("Cannot create test - missing organisation unit or tracked entity type")